#### Setting up the enviroment

In [ ]:
# cleaning directory
!rm -r *
# This command builds the special URL and passes it to aria2c
# -c = continue download
# -x 16 = max 16 connections per download
# -s 16 = split file into 16 pieces
# -k 1M = min split size
# -o = output file name
!apt install aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://zenodo.org/records/17252365/files/cadenza_clip1_data.train.v1.0.tar.gz?download=1" -o {OUTPUT_FILE}

In [ ]:
# This will extract the files into the current directory
!tar -xzvf cadenza_clip1_data.train.v1.0.tar.gz
!rm cadenza_clip1_data.train.v1.0.tar.gz
!ls -l cadenza_data 
!ls -l cadenza_data/metadata
!ls -l cadenza_data/train


In [ ]:
# Create the directory (ignore error if it already exists)
!mkdir -p baseline

# Run npx with -y to auto-confirm, and tell degit to download INTO the baseline folder
!npx -y degit --force claritychallenge/clarity/recipes/cad_icassp_2026/baseline#main baseline

#### Getting ready for running baselines

In [ ]:
import os
import sys

# All subsequent commands will be run from here.
os.chdir('baseline')
print(f"Current working directory: {os.getcwd()}")

# --- Install packages ---
print("\nInstalling required packages...")

# Install standard packages
!pip install -q hydra-core omegaconf pandas pystoi demucs openai-whisper jiwer inflect

# Install the pyclarity library directly from the official GitHub repository
print("\nInstalling pyclarity library from GitHub...")
!pip install -q git+https://github.com/claritychallenge/clarity.git

print("\nEnvironment setup complete.")

In [ ]:
import os

# just making this more structured
exp_dir = "exp"
os.makedirs(exp_dir, exist_ok=True)

local_precomputed_path = "precomputed"

# --- Copy STOI scores ---
print("Copying STOI scores...")
!cp {local_precomputed_path}/cadenza_data.train.stoi.jsonl {exp_dir}/
!cp {local_precomputed_path}/cadenza_data.valid.stoi.jsonl {exp_dir}/

# --- Copy Whisper scores ---
print("Copying Whisper scores...")
!cp {local_precomputed_path}/cadenza_data.train.whisper.jsonl {exp_dir}/
!cp {local_precomputed_path}/cadenza_data.valid.whisper.jsonl {exp_dir}/

print("\nPre-computed scores copied to the 'exp/' directory.")
!ls -l {exp_dir}

In [ ]:
import os
# just downloading the validation data.
os.chdir('/kaggle/working')

print("--- Downloading Validation Data ---")
# This is the same Zenodo record as the training data, just a different file
!aria2c -c -x 16 -s 16 -k 1M -o cadenza_clip1_data.valid.v1.0.tar.gz "https://zenodo.org/records/17252365/files/cadenza_clip1_data.valid.v1.0.tar.gz"

print("\n--- Extracting Validation Data ---")
!tar -xzvf cadenza_clip1_data.valid.v1.0.tar.gz
!rm cadenza_clip1_data.valid.v1.0.tar.gz

print("\nValidation data is now in place.")
!ls -l cadenza_data/metadata

# Return to our main working directory for the baseline code
os.chdir('baseline')
print(f"\nReturned to directory: {os.getcwd()}")

In [ ]:
# key issue fixing
print("--- Inspecting first line of the whisper score file ---")
!head -n 1 exp/cadenza_data.train.whisper.jsonl

# The key is "whisper_mixture". We need to replace it with "whisper".
print('\n--- Correcting the key in whisper.jsonl files ---')
!sed -i 's/"whisper.mixture"/"whisper"/g' exp/cadenza_data.train.whisper.jsonl
!sed -i 's/"whisper.mixture"/"whisper"/g' exp/cadenza_data.valid.whisper.jsonl

print("Keys have been corrected. Verifying the change:")
!head -n 1 exp/cadenza_data.train.whisper.jsonl

In [ ]:
# --- Run STOI baseline prediction ---
print("--- Generating predictions for STOI baseline ---")
!python predict.py baseline=stoi data.cadenza_data_root=/kaggle/working
print("\nSTOI prediction complete.")

# --- Run Whisper baseline prediction ---
print("\n--- Generating predictions for Whisper baseline ---")
!python predict.py baseline=whisper data.cadenza_data_root=/kaggle/working
print("\nWhisper prediction complete.")

Now you should have the computed csv files.

#### Now we shall be trying to run whisper large v3 turbo as a baseline too for further analysis

In [ ]:
%%bash
# Create the audio directory if it doesn't exist
mkdir -p /kaggle/working/cadenza_data/audio

# Remove any existing broken symlinks (ignore errors if they don't exist)
rm -f /kaggle/working/cadenza_data/audio/train
rm -f /kaggle/working/cadenza_data/audio/valid

# Create correct symlinks pointing to the actual data location
ln -s /kaggle/working/cadenza_data/train /content/cadenza_data/audio/train
ln -s /kaggle/working/cadenza_data/valid /content/cadenza_data/audio/valid

# Verify the new symlinks
echo "Verifying new symlinks:"
ls -la /kaggle/working/cadenza_data/audio/
echo ""
echo "Checking if we can access the files:"
ls /kaggle/working/cadenza_data/audio/train/signals/ | head -5

In [ ]:
!cd /kaggle/working/baseline/ && python compute_whisper.py \
  split=train \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.whisper_version=large-v3-turbo \
  baseline.system=whisper-large-v3-turbo

In [ ]:
!cd /kaggle/working/baseline/ && python compute_whisper.py \
  split=valid \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.whisper_version=large-v3-turbo \
  baseline.system=whisper-large-v3-turbo

In [ ]:
# You should see 'cadenza_data.train.whisper_large.jsonl' and 
# 'cadenza_data.valid.whisper_large.jsonl' in the output list.
!ls -lh /kaggle/working/baseline/exp/

In [ ]:
# Execute predict.py using the scores from your new 'whisper-large-v3-turbo' system.
# Note that the split is 'valid' because we are making predictions on the validation set.
!cd /kaggle/working/baseline/ && python predict.py \
  split=valid \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.system=whisper-large-v3-turbo

In [ ]:
# now you should see the new prediction file.
!ls -lh /kaggle/working/baseline/exp/

In [ ]:
# Run the evaluation script on the prediction file so we now know the rmse and corelation
!cd /kaggle/working/baseline/ && python evaluate.py \
  split=valid \
  data.cadenza_data_root=/content/ \
  baseline=whisper \
  baseline.system=whisper-large-v3-turbo